# Adaptive Practice Recommender — Data Pipeline & Model Training

This notebook builds a model that predicts whether a student will answer a question correctly, based on their past practice history. It uses a 1-million-row sample from the RIIID/EdNet dataset of real student-question interactions.

In [1]:
import pandas as pd

 Load the raw data

We only load the first 1 million rows instead of the full 100M+ row dataset, to keep things fast while prototyping. Each row is one interaction: a student either answering a question or watching a lecture.

In [2]:
df = pd.read_csv("train.csv", nrows=1000000)
df.shape

(1000000, 10)

In [3]:
df.head()

,row_id,timestamp,user_id,content_id,content_type_id,task_container_id,user_answer,answered_correctly,prior_question_elapsed_time,prior_question_had_explanation
0,0,0,115,5692,0,1,3,1,NaN,NaN
1,1,56943,115,5716,0,2,2,1,37000.0,False
2,2,118363,115,128,0,0,0,1,55000.0,False
3,3,131167,115,7860,0,3,0,1,19000.0,False
4,4,137965,115,7922,0,4,1,1,11000.0,False


Separate questions from lectures

The dataset mixes two types of rows: students answering questions, and students watching lectures. `content_type_id` tells them apart (0 = question, 1 = lecture). Since we're predicting answer correctness, we only need the question rows.

In [4]:
df['content_type_id'].value_counts()

content_type_id
0    980093
1     19907
Name: count, dtype: int64

In [5]:
df = df[df['content_type_id'] == 0]
df.shape

(980093, 10)

Add question metadata

`questions.csv` has extra information about each question, most importantly `part` — which subject/section the question belongs to. We merge it in so every interaction row also knows which topic (part) it was testing.

In [7]:
questions = pd.read_csv("questions.csv")
df = df.merge(questions, left_on='content_id', right_on='question_id')
df.head()

,row_id,timestamp,user_id,content_id,content_type_id,task_container_id,user_answer,answered_correctly,prior_question_elapsed_time,prior_question_had_explanation,question_id,bundle_id,correct_answer,part,tags
0,0,0,115,5692,0,1,3,1,NaN,NaN,5692,5692,3,5,151
1,1,56943,115,5716,0,2,2,1,37000.0,False,5716,5716,2,5,168
2,2,118363,115,128,0,0,0,1,55000.0,False,128,128,0,1,131 149 92
3,3,131167,115,7860,0,3,0,1,19000.0,False,7860,7860,0,1,131 104 81
4,4,137965,115,7922,0,4,1,1,11000.0,False,7922,7922,1,1,131 149 92


Sort by student and time

To calculate "how a student was doing before this attempt," we need every student's rows in chronological order. Without sorting, rolling calculations like cumulative accuracy would be meaningless.

In [8]:
df = df.sort_values(['user_id', 'timestamp'])

Feature 1 — Topic Accuracy

For every attempt, we calculate how accurate the student has been on this topic (`part`) *before* this attempt — using only past data, never the current or future answer, to avoid leaking the answer into the model.

- `attempt_before`: how many times the student has attempted this topic before now
- `correct_before`: how many of those past attempts were correct
- `topic_accuracy`: correct_before ÷ attempt_before

For a student's very first attempt on a topic (0 prior attempts), this division is undefined, so we fill it with 0.5 — a neutral "unknown" starting point.

In [10]:
df['attempt_before'] = df.groupby(['user_id', 'part']).cumcount()
df['correct_before'] = df.groupby(['user_id', 'part'])['answered_correctly'].cumsum() - df['answered_correctly']
df['topic_accuracy'] = df['correct_before'] / df['attempt_before']
df['topic_accuracy'] = df['topic_accuracy'].fillna(0.5)
df[['user_id', 'part', 'attempt_before', 'correct_before', 'topic_accuracy']].head(10)

,user_id,part,attempt_before,correct_before,topic_accuracy
0,115,5,0,0,0.5
1,115,5,1,1,1.0
2,115,1,0,0,0.5
3,115,1,1,1,1.0
4,115,1,2,2,1.0
5,115,1,3,3,1.0
6,115,1,4,4,1.0
7,115,1,5,5,1.0
8,115,1,6,6,1.0
9,115,1,7,7,1.0


 Feature 2 — Time Since Last Attempt

Memory fades with time — a student who practiced a topic yesterday is likely sharper on it than one who practiced it a month ago. This feature measures the time gap since the student's last attempt on the same topic.

For a student's first-ever attempt on a topic, there's no previous attempt to compare to, so we fill that gap with the maximum value in the column — treating "no prior attempt" as "maximum staleness."

In [11]:
df['time_since_last_attempt'] = df.groupby(['user_id', 'part'])['timestamp'].diff()
df['time_since_last_attempt'] = df['time_since_last_attempt'].fillna(df['time_since_last_attempt'].max())
df[['user_id', 'part', 'timestamp', 'time_since_last_attempt']].head(10)

,user_id,part,timestamp,time_since_last_attempt
0,115,5,0,6.868745e+10
1,115,5,56943,5.694300e+04
2,115,1,118363,6.868745e+10
3,115,1,131167,1.280400e+04
4,115,1,137965,6.798000e+03
5,115,1,157063,1.909800e+04
6,115,1,176092,1.902900e+04
7,115,1,194190,1.809800e+04
8,115,1,212463,1.827300e+04
9,115,1,230983,1.852000e+04


Feature 3 — Question Difficulty

Some questions are just harder than others, regardless of the student. We estimate a question's difficulty as the average correctness rate across *all* students who've answered it.

**Known limitation:** this average is computed using the entire dataset (not just training data), which introduces a small amount of data leakage. This is a deliberate simplification for this project's scope — a production system would compute it from training data only.

In [12]:
df['ques_avg_accuracy'] = df.groupby('content_id')['answered_correctly'].transform('mean')

Build the feature matrix

`X` holds our three engineered features (the inputs). `y` holds what we're trying to predict — whether the answer was correct (1) or not (0).

In [13]:
X = df[['topic_accuracy', 'time_since_last_attempt', 'ques_avg_accuracy']]
y = df['answered_correctly']
X.shape, y.shape

((980093, 3), (980093,))

# Train/test split

We hold out 20% of the data as a test set — the model never sees this during training, so it gives us an honest read on how well the model generalizes to new data.

In [14]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((784074, 3), (196019, 3))

Baseline check

Before trusting any model, we check what a "lazy guesser" would score by always predicting the majority class. Any real model needs to beat this baseline to prove it's actually learning something.

In [16]:
y_test.value_counts()

answered_correctly
1    127612
0     68407
Name: count, dtype: int64

##  Model 1 — Logistic Regression (unscaled features)

A simple linear model, trained directly on the raw feature values (no scaling). One of our features (`time_since_last_attempt`) is in a much larger numeric range than the others, which tends to break linear models like this one.

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model_lr = LogisticRegression()
model_lr.fit(X_train, y_train)
y_pred = model_lr.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.6510185237145378


## 
 Model 2 — Logistic Regression (scaled features)

`StandardScaler` rescales every feature to a comparable range (mean 0, similar spread), which linear models need to actually use all the features meaningfully. Note: the scaler is fit only on training data, then applied to test data — fitting it on test data too would leak test-set information into training.

In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_lr_scaled = LogisticRegression()
model_lr_scaled.fit(X_train_scaled, y_train)
scaled_y_pred = model_lr_scaled.predict(X_test_scaled)
accuracy_scaled = accuracy_score(y_test, scaled_y_pred)
print(accuracy_scaled)

0.7101097342604543


## Model 3 — Random Forest

An ensemble of decision trees. Unlike logistic regression, tree-based models aren't sensitive to feature scale, but we reuse the scaled features here for a fair, consistent comparison across models.

In [27]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(random_state=42)
model_rf.fit(X_train_scaled, y_train)
y_pred_rf = model_rf.predict(X_test_scaled)
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(accuracy_rf)

0.6963253562154689


## Model 4 — XGBoost

A gradient-boosted tree model — typically one of the strongest performers on structured/tabular data like this.

In [26]:
from xgboost import XGBClassifier

model_xgb = XGBClassifier(random_state= 42)
model_xgb.fit(X_train_scaled, y_train)
y_pred_xgb = model_xgb.predict(X_test_scaled)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
print(accuracy_xgb)

0.7177977645024207


Save the winning model

XGBoost had the best accuracy of the four, so we save it to disk with `joblib`. This is the exact file the FastAPI backend loads to make live predictions.

In [21]:
import joblib
joblib.dump(model_xgb, "model_xgb.pkl")

['model_xgb.pkl']

Build a per-student snapshot table

For the live recommendation feature, we don't need a student's entire history — just their *latest* feature values for each topic. We take the last row per (student, topic) pair.

In [22]:
student_topic_summary = df.groupby(['user_id', 'part']).tail(1)[
    ['user_id', 'part', 'topic_accuracy', 'time_since_last_attempt', 'ques_avg_accuracy']
]
student_topic_summary.shape

(17641, 5)

In [23]:
student_topic_summary.to_csv("student_topic_summary.csv", index=False)

##  Recommendation function

Given a student ID, this looks up their latest features per topic, asks the model for the probability of answering correctly on each, and ranks topics from weakest (lowest probability) to strongest — the exact "study this first" order shown in the app.

In [24]:
def recommend_topics(user_id):
    student_data = student_topic_summary[student_topic_summary['user_id'] == user_id]
    features = student_data[['topic_accuracy', 'time_since_last_attempt', 'ques_avg_accuracy']]
    probabilities = model_xgb.predict_proba(features)[:, 1]
    student_data = student_data.copy()
    student_data['predicted_prob'] = probabilities
    ranked = student_data.sort_values('predicted_prob')
    return ranked[['part', 'predicted_prob']]

Test it end to end

A sanity check with a real student ID — this should match the same output the live API returns.

In [25]:
recommend_topics(115)

,part,predicted_prob
45,4,0.689122
39,2,0.748576
38,1,0.785765
42,3,0.830619
1,5,0.856599
